# Train nhiều model với nhiều bộ tham số khác nhau và so sánh Loss

In [1]:
# Multi-Model Training and Evaluation Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import tensorflow as tf
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
import os, sys, gc, time, warnings, pickle, psutil, random
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math
from lightgbm import early_stopping, log_evaluation
import tracemalloc

print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

# Load and prepare data (keeping your original preprocessing)
df = pd.read_parquet('../../data/train_val.parquet')
print(df.columns)
print(df['disaster_severity_flag'].unique())
df = df.drop(['date','disaster_type'],axis=1)

def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
   
    for i, t in enumerate(types):
        col = cols[i]
        if 'int' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif 'float' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)    
    return df

sample_submission = pd.read_csv('../../data/sample_submission.csv')
sample_submission = downcast(sample_submission)

# Constants
LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942
LENGTH = 28

def demand_features_eval(df):
    """ Same as demand_features but for the step-by-step evaluation """
    out = df.groupby('id', sort=False).last()
    for lag in LAGS:
        out[f'lag_t{lag}'] = df.groupby('id', sort=False)['demand'].nth(-lag-1).astype("float32")
        for w in WINDOWS:
            out[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id', sort=False)['demand'].nth(list(range(-lag-w, -lag))).groupby('id', sort=False).mean().astype("float32")
   
    return out.reset_index()

def prep_train(df, drop_d=1000):
    x = list(set(df.columns) - {'id', 'd', 'demand'})
    print(f"Features: {x}")
    
    # Split into test, valid, train
    test = df[df.d >= FIRST - max(LAGS) - max(WINDOWS) - 28]
    df = df[df.d < FIRST]
    xtrain, xvalid, ytrain, yvalid = train_test_split(df[x], df["demand"], test_size=0.1, shuffle=True, random_state=54)
    
    return xtrain, xvalid, ytrain, yvalid, test, x

# Metric calculation functions
def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def calculate_metrics(y_true, y_pred):
    """Calculate all metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'MAPE': mape}

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024


Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Index(['id', 'item_id', 'store_id', 'state_id', 'dept_id', 'cat_id', 'd',
       'demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28',
       'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'date',
       'wday', 'month', 'year', 'event_name_1', 'event_name_2', 'snap_CA',
       'snap_TX', 'snap_WI', 'sell_price', 'disaster_type',
       'disaster_severity_flag', 'flag_ufc', 'gold_price', 'humidity'],
      dtype='object')
[ 0 15  3  7 11]


In [2]:
# Model fitting functions
def fit_lightgbm(xtrain, ytrain, xvalid, yvalid, objective='regression'):
    """Fit LightGBM model"""
    if objective == 'regression':
        params = {
            'metric': 'rmse',
            'objective': 'regression',
            'seed': 200,
            'force_row_wise': True,
            'learning_rate': 0.05303667024743503,
            'lambda_l1': 0.002354253988281785,
            'lambda_l2': 7.053121220857246e-05,
            'num_leaves': 1601,
            'feature_fraction': 0.9834112644063673,
            'bagging_fraction': 0.9606769151362458,
            'bagging_freq': 2,
            'min_child_samples': 30,
        }
    elif objective == 'mae':
        params = {
            'metric': 'mae',
            'objective': 'regression_l1',
            'seed': 200,
            'force_row_wise': True,
            'learning_rate': 0.05,
            'num_leaves': 1601,
            'feature_fraction': 0.98,
            'bagging_fraction': 0.96,
            'bagging_freq': 2,
            'min_child_samples': 30,
        }
    else:  # mape
        params = {
            'metric': 'mape',
            'objective': 'mape',
            'seed': 200,
            'force_row_wise': True,
            'learning_rate': 0.05,
            'num_leaves': 1601,
            'feature_fraction': 0.98,
            'bagging_fraction': 0.96,
            'bagging_freq': 2,
            'min_child_samples': 30,
        }
    
    train_data = lgb.Dataset(xtrain, label=ytrain)
    valid_data = lgb.Dataset(xvalid, label=yvalid)
    
    callbacks = [
        early_stopping(stopping_rounds=100),
        log_evaluation(period=0)
    ]
    
    model = lgb.train(params,
                     train_data,
                     num_boost_round=200,
                     valid_sets=[valid_data],
                     callbacks=callbacks)
    return model

def fit_xgboost(xtrain, ytrain, xvalid, yvalid, objective='reg:squarederror'):
    """Fit XGBoost model"""
    if objective == 'reg:squarederror':
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'seed': 200,
            'learning_rate': 0.05,
            'max_depth': 8,
            'subsample': 0.96,
            'colsample_bytree': 0.98,
            'min_child_weight': 30,
        }
    elif objective == 'reg:absoluteerror':
        params = {
            'objective': 'reg:absoluteerror',
            'eval_metric': 'mae',
            'seed': 200,
            'learning_rate': 0.05,
            'max_depth': 8,
            'subsample': 0.96,
            'colsample_bytree': 0.98,
            'min_child_weight': 30,
        }
    else:  # reg:pseudohubererror for MAPE-like
        params = {
            'objective': 'reg:pseudohubererror',
            'eval_metric': 'rmse',
            'seed': 200,
            'learning_rate': 0.05,
            'max_depth': 8,
            'subsample': 0.96,
            'colsample_bytree': 0.98,
            'min_child_weight': 30,
        }
    
    dtrain = xgb.DMatrix(xtrain, label=ytrain)
    dvalid = xgb.DMatrix(xvalid, label=yvalid)
    
    model = xgb.train(params,
                     dtrain,
                     num_boost_round=200,
                     evals=[(dvalid, 'valid')],
                     early_stopping_rounds=100,
                     verbose_eval=0)
    return model

def fit_catboost(xtrain, ytrain, xvalid, yvalid, loss_function='RMSE'):
    """Fit CatBoost model"""
    params = {
        'loss_function': loss_function,
        'eval_metric': loss_function,
        'random_seed': 200,
        'learning_rate': 0.05,
        'depth': 8,
        'subsample': 0.96,
        'colsample_bylevel': 0.98,
        'min_child_samples': 30,
        'verbose': False
    }
    
    model = cb.CatBoostRegressor(**params)
    model.fit(xtrain, ytrain,
             eval_set=(xvalid, yvalid),
             early_stopping_rounds=100,
             verbose=False)
    return model

def fit_randomforest(xtrain, ytrain, criterion='squared_error'):
    """Fit Random Forest model"""
    params = {
        'n_estimators': 200,
        'max_depth': 15,
        'min_samples_split': 10,
        'min_samples_leaf': 5,
        'random_state': 200,
        'criterion': criterion,
        'n_jobs': -1
    }
    
    model = RandomForestRegressor(**params)
    model.fit(xtrain, ytrain)
    return model

def create_ensemble(models, xtrain, ytrain):
    """Create ensemble model"""
    ensemble = VotingRegressor(estimators=models)
    ensemble.fit(xtrain, ytrain)
    return ensemble

# Prediction functions
def predict_model(model, X, model_type='lightgbm'):
    """Make predictions based on model type"""
    if model_type == 'lightgbm':
        return model.predict(X)
    elif model_type == 'xgboost':
        dX = xgb.DMatrix(X)
        return model.predict(dX)
    elif model_type == 'catboost':
        return model.predict(X)
    elif model_type == 'randomforest':
        return model.predict(X)
    elif model_type == 'ensemble':
        return model.predict(X)

# Main training and evaluation function
def train_and_evaluate_models(xtrain, ytrain, xvalid, yvalid):
    """Train and evaluate all models with all loss functions"""
    
    results = []
    
    # Define models and their corresponding loss functions
    model_configs = {
        'LightGBM': {
            'objectives': ['regression', 'mae', 'mape'],
            'fit_func': fit_lightgbm,
            'model_type': 'lightgbm'
        },
        'XGBoost': {
            'objectives': ['reg:squarederror', 'reg:absoluteerror', 'reg:pseudohubererror'],
            'fit_func': fit_xgboost,
            'model_type': 'xgboost'
        },
        'CatBoost': {
            'objectives': ['RMSE', 'MAE', 'MAPE'],
            'fit_func': fit_catboost,
            'model_type': 'catboost'
        },
        'RandomForest': {
            'objectives': ['squared_error', 'absolute_error', 'friedman_mse'],
            'fit_func': fit_randomforest,
            'model_type': 'randomforest'
        }
    }
    
    trained_models = {}
    
    for model_name, config in model_configs.items():
        for i, objective in enumerate(config['objectives']):
            loss_name = ['MSE', 'MAE', 'MAPE'][i]
            
            print(f"\nTraining {model_name} with {loss_name} loss...")
            
            # Start memory and time tracking
            tracemalloc.start()
            start_time = time.time()
            start_memory = get_memory_usage()
            
            try:
                # Fit model
                if model_name == 'RandomForest':
                    model = config['fit_func'](xtrain, ytrain, objective)
                else:
                    model = config['fit_func'](xtrain, ytrain, xvalid, yvalid, objective)
                
                # Make predictions
                y_pred = predict_model(model, xvalid, config['model_type'])
                
                # Calculate metrics
                metrics = calculate_metrics(yvalid, y_pred)
                
                # Calculate time and memory
                end_time = time.time()
                end_memory = get_memory_usage()
                current, peak = tracemalloc.get_traced_memory()
                tracemalloc.stop()
                
                training_time = end_time - start_time
                memory_used = end_memory - start_memory
                peak_memory = peak / 1024 / 1024  # Convert to MB
                
                # Store results
                result = {
                    'Model': model_name,
                    'Loss_Function': loss_name,
                    'MAE': metrics['MAE'],
                    'MSE': metrics['MSE'],
                    'MAPE': metrics['MAPE'],
                    'Training_Time(s)': training_time,
                    'Memory_Used(MB)': memory_used,
                    'Peak_Memory(MB)': peak_memory
                }
                
                results.append(result)
                trained_models[f"{model_name}_{loss_name}"] = model
                
                print(f"✅ {model_name} ({loss_name}) - MAE: {metrics['MAE']:.4f}, MSE: {metrics['MSE']:.4f}, MAPE: {metrics['MAPE']:.2f}%")
                print(f"   Time: {training_time:.2f}s, Memory: {memory_used:.2f}MB, Peak: {peak_memory:.2f}MB")
                
            except Exception as e:
                print(f"❌ Error training {model_name} with {loss_name}: {str(e)}")
                continue
    
    # Create ensemble models
    print("\nCreating ensemble models...")
    for loss_name in ['MSE', 'MAE', 'MAPE']:
        try:
            # Get models for this loss function
            models_for_ensemble = []
            for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'RandomForest']:
                model_key = f"{model_name}_{loss_name}"
                if model_key in trained_models:
                    models_for_ensemble.append((model_name.lower(), trained_models[model_key]))
            
            if len(models_for_ensemble) >= 2:
                tracemalloc.start()
                start_time = time.time()
                start_memory = get_memory_usage()
                
                # Create ensemble
                ensemble = create_ensemble(models_for_ensemble, xtrain, ytrain)
                
                # Make predictions
                y_pred = predict_model(ensemble, xvalid, 'ensemble')
                
                # Calculate metrics
                metrics = calculate_metrics(yvalid, y_pred)
                
                # Calculate time and memory
                end_time = time.time()
                end_memory = get_memory_usage()
                current, peak = tracemalloc.get_traced_memory()
                tracemalloc.stop()
                
                training_time = end_time - start_time
                memory_used = end_memory - start_memory
                peak_memory = peak / 1024 / 1024
                
                result = {
                    'Model': 'Ensemble',
                    'Loss_Function': loss_name,
                    'MAE': metrics['MAE'],
                    'MSE': metrics['MSE'],
                    'MAPE': metrics['MAPE'],
                    'Training_Time(s)': training_time,
                    'Memory_Used(MB)': memory_used,
                    'Peak_Memory(MB)': peak_memory
                }
                
                results.append(result)
                trained_models[f"Ensemble_{loss_name}"] = ensemble
                
                print(f"✅ Ensemble ({loss_name}) - MAE: {metrics['MAE']:.4f}, MSE: {metrics['MSE']:.4f}, MAPE: {metrics['MAPE']:.2f}%")
                
        except Exception as e:
            print(f"❌ Error creating ensemble for {loss_name}: {str(e)}")
            continue
    
    return pd.DataFrame(results), trained_models

In [ ]:
# Prepare data
xtrain, xvalid, ytrain, yvalid, test, x = prep_train(df, 1000 - 28)
del df
gc.collect()

print("Starting model training and evaluation...")
results_df, trained_models = train_and_evaluate_models(xtrain, ytrain, xvalid, yvalid)

# Display results
print("\n" + "="*100)
print("FINAL RESULTS SUMMARY")
print("="*100)
print(results_df.to_string(index=False))

# Find best models for each metric
print("\n" + "="*50)
print("BEST MODELS BY METRIC")
print("="*50)

for metric in ['MAE', 'MSE', 'MAPE']:
    best_idx = results_df[metric].idxmin()
    best_model = results_df.iloc[best_idx]
    print(f"\nBest {metric}: {best_model['Model']} ({best_model['Loss_Function']})")
    print(f"  {metric}: {best_model[metric]:.4f}")
    print(f"  Training Time: {best_model['Training_Time(s)']:.2f}s")
    print(f"  Memory Used: {best_model['Memory_Used(MB)']:.2f}MB")

# Save results
results_df.to_csv('model_comparison_results.csv', index=False)
print(f"\n📊 Results saved to 'model_comparison_results.csv'")

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot metrics
for i, metric in enumerate(['MAE', 'MSE', 'MAPE']):
    if i < 3:
        ax = axes[i//2, i%2]
        pivot_data = results_df.pivot(index='Model', columns='Loss_Function', values=metric)
        pivot_data.plot(kind='bar', ax=ax, title=f'{metric} Comparison')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=45)

# Plot training time
ax = axes[1, 1]
pivot_time = results_df.pivot(index='Model', columns='Loss_Function', values='Training_Time(s)')
pivot_time.plot(kind='bar', ax=ax, title='Training Time Comparison')
ax.set_ylabel('Time (seconds)')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('model_comparison_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("📈 Visualization saved to 'model_comparison_plots.png'")

Features: ['snap_WI', 'dept_id', 'rolling_mean_lag28_w7', 'year', 'disaster_severity_flag', 'cat_id', 'lag_t7', 'rolling_mean_lag7_w28', 'rolling_mean_lag28_w28', 'event_name_1', 'snap_TX', 'snap_CA', 'state_id', 'sell_price', 'event_name_2', 'gold_price', 'lag_t28', 'month', 'item_id', 'wday', 'flag_ufc', 'store_id', 'humidity', 'rolling_mean_lag7_w7']
Starting model training and evaluation...

Training LightGBM with MSE loss...
[LightGBM] [Info] Total Bins 2296
[LightGBM] [Info] Number of data points in the train set: 25053633, number of used features: 24
[LightGBM] [Info] Start training from score 1.228829
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[200]	valid_0's rmse: 2.09153
✅ LightGBM (MSE) - MAE: 0.9138, MSE: 4.3745, MAPE: 57.06%
   Time: 242.37s, Memory: -1710.86MB, Peak: 4587.69MB

Training LightGBM with MAE loss...
[LightGBM] [Info] Total Bins 2296
[LightGBM] [Info] Number of data points in the train set: 250